In [ ]:
from gym_interface import Agent, State
import copy
import math
import random
import socket
import time
from collections import deque, namedtuple
from typing import Dict, Iterable, List, Literal, Optional, Union

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from distutils.util import strtobool
from rlmodel.utils.utils import print_args, print_box, connected_to_internet
import wandb
import setproctitle
from pathlib import Path

import os, sys
from rlmodel.onpolicy.runner.shared.mpe_runner import MPERunner as Runner
from concurrent.futures import ThreadPoolExecutor

*自定义处理函数*

In [ ]:
def parse_Input(action: int) -> str:
    #example:
    if(action == -1):
        return ''
    TacticID = f'<AgentCMD><uint64_t>{action}</uint64_t></AgentCMD>'
    s = '<c>' + TacticID + '</c>'
    # s = {
    #     "AgentCMD": TacticID
    # }
    return s

In [ ]:
def parse_Output(state: Dict[str, any]) -> dict:
    #example:
    tmp = []
    Input = {}
    for input in state:
        for k, v in input.items():
            if k == 'PlaneInfo':
                tmp.append(v)
            else:
                Input[k] = v
    Input['PlaneInfo'] = tmp
    return Input

In [ ]:
def outputToTensor(state: Dict[str, any]) -> np.array:
    self = state['PlaneInfo'][0]['entity_motion_state']
    enemy = state['TargetMessage']['entity_motion_state']
    self_pitch = self['AttitudeInfo']['pitch']/10
    self_yaw = self['AttitudeInfo']['yaw']/10
    self_roll = self['AttitudeInfo']['roll']/10
    self_lon = self['PositionInfo']['longitude']/10
    self_lat = self['PositionInfo']['latitude']/10
    self_alt = self['PositionInfo']['altitude']/100
    self_vx = self['ECEFVelocity']['vx']/10
    self_vy = self['ECEFVelocity']['vy']/10
    self_vz = self['ECEFVelocity']['vz']/10
    enemy_pitch = enemy['AttitudeInfo']['pitch']/10
    enemy_yaw = enemy['AttitudeInfo']['yaw']/10
    enemy_roll = enemy['AttitudeInfo']['roll']/10
    enemy_lon = enemy['PositionInfo']['longitude']/10
    enemy_lat = enemy['PositionInfo']['latitude']/10
    enemy_alt = enemy['PositionInfo']['altitude']/100
    enemy_vx = enemy['ECEFVelocity']['vx']/10
    enemy_vy = enemy['ECEFVelocity']['vy']/10
    enemy_vz = enemy['ECEFVelocity']['vz']/10
    return np.array([self_pitch, self_yaw, self_roll, self_lon, 
                     self_lat, self_alt, self_vx, self_vy, self_vz, 
                     enemy_pitch, enemy_yaw, enemy_roll, enemy_lon, 
                     enemy_lat, enemy_alt, enemy_vx, enemy_vy, enemy_vz])

In [ ]:
def cal_Reward(state:Dict[str, any]):
    #example:
    if not state["PlaneInfo"]:
        return 0
    self = state['PlaneInfo'][0]['entity_motion_state']
    enemy = state['TargetMessage']['entity_motion_state']
    self_lon = self['PositionInfo']['longitude']/10
    self_lat = self['PositionInfo']['latitude']/10
    self_alt = self['PositionInfo']['altitude']/100
    enemy_lon = enemy['PositionInfo']['longitude']/10
    enemy_lat = enemy['PositionInfo']['latitude']/10
    enemy_alt = enemy['PositionInfo']['altitude']/100
    self_state = state['PlaneInfo'][0]['entity_state']
    enemy_state = state['TargetMessage']['entity_state']
    dist_to_goal = - np.sqrt(np.max(np.square(self_lon-enemy_lon), 0) + np.max(np.square(self_lat-enemy_lat), 0) + np.max(np.square(self_alt-enemy_alt), 0))
    r = dist_to_goal
    if(enemy_state == 5):
        print("敌死")
        r += 10000
    elif(self_state == 5):
        print("我亡")
        r -= 10000
    return r

In [ ]:
def cal_Termination(state:Dict[str, any]) -> bool:
    #example
        self = state['PlaneInfo'][0]['entity_state']
        enemy = state['TargetMessage']['entity_state']
        odds = np.random.randint(0,10)
        if self == 5 or enemy == 5:
            return True
        else:    
            return False


_算法参数配置_

In [ ]:
device = torch.device("cuda")
sys.path.append(os.path.abspath(os.getcwd()))
num_agents = 1
num_enemies = 1
episode_length = 10
save_interval = 1000
log_interval = 10
model_dir = (
        Path(os.path.split(os.getcwd())[0] + "/results")
        /"save"
    )
all_args = {
    "algorithm_name": "rmappo",
    "use_recurrent_policy": False,
    "use_naive_recurrent_policy": False,
    "share_policy": True,
    "use_wandb": True,
    "seed": 0,
    "use_centralized_V": True,
    "use_linear_lr_decay": True,
    "hidden_size": 16,
    "recurrent_N": 1,
    "act_space": 5,
    "obs_space": 18,
    "shared_obs_space": 18*num_agents,
    "model_dir": None,
    "episode_length": episode_length,
    "gamma": 0.98,
    "gae_lambda": 0.95,
    "use_gae": True,
    "clip_param": 0.2,
    "ppo_epoch": 15,
    "num_mini_batch": 1,
    "data_chunk_length": 10,
    "value_loss_coef": 0.5,
    "entropy_coef": 0.01,
    "max_grad_norm": 10.0,
    "huber_delta": 10.0,
    "use_max_grad_norm": True,
    "use_clipped_value_loss": True,
    "use_huber_loss": True,
    "use_popart": True,
    "use_valuenorm": False,
    "use_value_active_masks": True,
    "use_policy_active_masks": True,
    "lr": 7e-3,
    "critic_lr": 7e-4,
    "opti_eps": 1e-5,
    "weight_decay": 0,
    "gain": 0.01,
    "use_orthogonal": True,
    "use_feature_normalization": True,
    "use_ReLU": False,
    "stacked_frames": 1,
    "layer_N": 1,
    "n_rollout_threads": 2,
}


run_dir = (
        Path(os.path.split(os.getcwd())[0] + "/results")
        / all_args["algorithm_name"]
    )
config = {
    "all_args": all_args,
    "num_agents": num_agents,
    "num_enemies":num_enemies,
    "device": device,
    "run_dir": run_dir
}

*W&B记录训练日志*

In [ ]:
if all_args["use_wandb"]:
        # for supercloud when no internet_connection
        if not connected_to_internet():
            import json

            # save a json file with your wandb api key in your
            # home folder as {'my_wandb_api_key': 'INSERT API HERE'}
            # NOTE this is only for running on systems without internet access
            # have to run `wandb sync wandb/run_name` to sync logs to wandboard
            with open(os.path.split(os.getcwd())[0] + "/keys.json") as json_file:
                key = json.load(json_file)
                my_wandb_api_key = key["my_wandb_api_key"]  # NOTE change here as well
            os.environ["WANDB_API_KEY"] = my_wandb_api_key
            os.environ["WANDB_MODE"] = "dryrun"
            os.environ["WANDB_SAVE_CODE"] = "true"

        print_box("Creating wandboard...")
        run = wandb.init(
            config=all_args,
            project="simplecq",
            # project=all_args.env_name,
            # entity="cc",
            notes=socket.gethostname(),
            name=str(all_args["algorithm_name"])
            + "_seed"
            + str(all_args["seed"]),
            # group=all_args.scenario_name,
            dir=str(run_dir),
            # job_type="training",
            reinit=True,
        )
        
setproctitle.setproctitle(
        str(all_args["algorithm_name"])
        + "@"
        + str("lapluma030")
    )

# seed
torch.manual_seed(all_args["seed"])
torch.cuda.manual_seed_all(all_args["seed"])
np.random.seed(all_args["seed"])

In [ ]:
runner = Runner(config)
print_box("Actor Network", 80)
if type(runner.policy) == list:
    print_box(runner.policy[0].actor, 80)
    print_box("Critic Network", 80)
    print_box(runner.policy[0].critic, 80)
else:
    print_box(runner.policy.actor, 80)
    print_box("Critic Network", 80)
    print_box(runner.policy.critic, 80)



*端口及输出类型指定*

In [ ]:
port = 40029
outputs_type = "uint64_t"

In [ ]:
agents = []
for i in range(all_args["n_rollout_threads"]):
    agents.append(Agent(port=port+i, 
                outputs_type=outputs_type,
                process_input=parse_Input,
                process_output=parse_Output,
                reward_func=cal_Reward,
                end_func=cal_Termination))


*训练函数*

In [ ]:
def train(env: List[Agent], episodes, enable_log=True):
    Win = 0
    Lose = 0
    with ThreadPoolExecutor() as executor:
        tasks = [
            executor.submit(env[i].reset)
            for i in range(all_args["n_rollout_threads"])
        ]
        results = [task.result() for task in tasks]
    
    s = np.zeros((all_args["n_rollout_threads"] ,num_agents, all_args["obs_space"]))
    r = np.zeros((all_args["n_rollout_threads"] ,num_agents, 1))
    dones =  np.full((all_args["n_rollout_threads"], num_agents), False)

    # replay buffer
    if runner.use_centralized_V:
        share_obs = s.reshape(all_args["n_rollout_threads"], -1)
        share_obs = np.expand_dims(share_obs, 1).repeat(runner.num_agents, axis=1)
    else:
        share_obs = s

    runner.buffer.share_obs[0] = share_obs.copy()
    runner.buffer.obs[0] = s.copy()

    count = 0
    for episode in range(episodes):


        if all_args["use_linear_lr_decay"]:

            runner.trainer.policy.lr_decay(episode, episodes)


        for step in range(episode_length):
            # Sample actions
            if step == episode_length - 1:
                (
                    values,
                    actions,
                    action_log_probs,
                    rnn_states,
                    rnn_states_critic,
                    actions_env,
                ) = runner.collect(step)

                TacticList = [1, 10, 14, 27, 32]
                outputs = np.array(TacticList)
                with ThreadPoolExecutor() as executor:
                    tasks = [
                        executor.submit(env[i].step, outputs[actions[i, 0, 0]])
                        for i in range(all_args["n_rollout_threads"])
                    ]
                    results = [task.result() for task in tasks]  # 按顺序收集结果
                obs, rew, done , _= zip(*results)
                
                s = np.array([[outputToTensor(i) for _ in range(num_agents)] for i in obs])
                r = np.array([[[j] for _ in range(num_agents) ] for j in rew])
                dones = np.array([[ k for _ in range(num_agents) ] for k in done])

                data = (
                    s,
                    r,
                    dones,
                    values,
                    actions,
                    action_log_probs,
                    rnn_states,
                    rnn_states_critic,
                )

                # insert data into buffer
                runner.insert(data)
               
                if (np.any(dones) == True) or count > 10000:
                    count = 0
                    with ThreadPoolExecutor() as executor:
                        tasks = [
                            executor.submit(env[i].reset)
                            for i in range(all_args["n_rollout_threads"])
                        ]
                        results = [task.result() for task in tasks]
            else:
                with ThreadPoolExecutor() as executor:
                    tasks = [
                        executor.submit(env[i].step, -1)
                        for i in range(all_args["n_rollout_threads"])
                    ]
                    results = [task.result() for task in tasks]
                    _, _, done , _= zip(*results)

                dones =  np.array([[ k for _ in range(num_agents) ] for k in done])
                # obs, rew, done, _ = env.step(action=-1)
                if (np.any(dones) == True) or count > 10000:
                    count = 0
                    with ThreadPoolExecutor() as executor:
                        tasks = [
                            executor.submit(env[i].reset)
                            for i in range(all_args["n_rollout_threads"])
                        ]
                        results = [task.result() for task in tasks]

            if step == episode_length-1:
                runner.compute()
                train_infos = runner.train()
                
            # if obs['PlaneInfo'][0]['entity_state']==5:
            #     Win+=1
            # if obs['TargetMessage']['entity_state']==5: 
            #     Lose+=1
                
            count += 1

        # post process
        total_num_steps = (episode + 1) * episode_length * all_args["n_rollout_threads"]

        # save model
        if episode % save_interval == 0 or episode == episodes - 1:
            runner.save()

        # log information
        if episode % log_interval == 0:

            avg_ep_rew = np.mean(runner.buffer.rewards) * episode_length
            train_infos["average_episode_rewards"] = avg_ep_rew
            print(
                f"Average episode rewards is {avg_ep_rew:.3f} \t"
                f"Total timesteps: {total_num_steps} \t "
                f"Percentage complete {total_num_steps / episodes / episode_length / all_args['n_rollout_threads'] * 100:.3f}"
            )
            runner.log_train(train_infos, total_num_steps)


        if episode % 1000 == 0 and enable_log:
            print(f"finished: {episode / episodes * 100}%")

    print(f"Training is done. Win:{Win} Lose: {Lose}")
    if all_args["use_wandb"]:

        run.finish()

*运行训练*

In [ ]:
train(agents, 50000)